# 01 — Data Collection & Feature Engineering
Complete rebuild. 2010-2022 WC only. 10 team features + 5 player features.

Sources: Local CSVs + StatsBomb xG (2018, 2022)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from pathlib import Path
import requests
import time

DATA_DIR = Path('../data')
NAME_MAP = {
    'USA': 'United States',
    'Ivory Coast': 'Cote dIvoire',
    'Korea Republic': 'South Korea',
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
    'Türkiye': 'Turkey',
    'IR Iran': 'Iran',
    'Congo DR': 'DR Congo',
    'Czechia': 'Czech Republic',
    'United States of America': 'United States',
    "Côte d'Ivoire": 'Cote dIvoire',
    'Korea DPR': 'North Korea',
    'Democratic Republic of the Congo': 'DR Congo',
}

def normalise(name):
    return NAME_MAP.get(str(name).strip(), str(name).strip())

GROUPS_2026 = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['United States', 'Paraguay', 'Australia', 'Turkey'],
    'E': ['Germany', 'Cote dIvoire', 'Thailand', 'Chile'],
    'F': ['Spain', 'Egypt', 'Tunisia', 'Costa Rica'],
    'G': ['England', 'Croatia', 'Ghana', 'Panama'],
    'H': ['France', 'Senegal', 'Iraq', 'Norway'],
    'I': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'J': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'K': ['Netherlands', 'Iran', 'New Zealand', 'Japan'],
    'L': ['Belgium', 'Uruguay', 'Sweden', 'Saudi Arabia'],
}
ALL_TEAMS_2026 = sorted([t for teams in GROUPS_2026.values() for t in teams])
print(f'Setup: {len(ALL_TEAMS_2026)} teams')

## 1. Load & filter (2010, 2014, 2018, 2022 only)

In [ ]:
matches_raw = pd.read_csv(DATA_DIR / 'matches.csv')
goals_raw = pd.read_csv(DATA_DIR / 'goals.csv')
bookings_raw = pd.read_csv(DATA_DIR / 'bookings.csv')

for df in [matches_raw, goals_raw, bookings_raw]:
    for col in df.columns:
        if 'team' in col.lower() and df[col].dtype == object:
            df[col] = df[col].apply(normalise)

if 'year' not in matches_raw.columns and 'match_date' in matches_raw.columns:
    matches_raw['year'] = pd.to_datetime(matches_raw['match_date']).dt.year

VALID_YEARS = {2010, 2014, 2018, 2022}
matches = matches_raw[matches_raw['year'].isin(VALID_YEARS)].copy()
goals = goals_raw[goals_raw['year'].isin(VALID_YEARS)].copy()
bookings = bookings_raw[bookings_raw['year'].isin(VALID_YEARS)].copy()

print(f'Matches: {len(matches)}, Goals: {len(goals)}, Bookings: {len(bookings)')

## 2. StatsBomb xG (2018 + 2022)

In [ ]:
STATSBOMB_BASE = 'https://raw.githubusercontent.com/statsbomb/open-data/master/data/'

def get_sb_matches(comp_id, season_id):
    url = f'{STATSBOMB_BASE}matches/{comp_id}/{season_id}.json'
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def get_sb_events(mid):
    url = f'{STATSBOMB_BASE}events/{mid}.json'
    r = requests.get(url, timeout=30)
    return r.json() if r.status_code == 200 else []

sb_2018 = get_sb_matches(43, 3)
sb_2022 = get_sb_matches(43, 106)
print(f'StatsBomb 2018: {len(sb_2018)}, 2022: {len(sb_2022)}')

In [ ]:
def extract_xg(sb_list, year):
    rows = []
    for m in sb_list:
        home = normalise(m['home_team']['home_team_name'])
        away = normalise(m['away_team']['away_team_name'])
        events = get_sb_events(m['match_id'])
        h_xg = a_xg = 0.0
        for e in events:
            if e.get('type', {}).get('name') == 'Shot':
                xg = e.get('shot', {}).get('statsbomb_xg', 0) or 0
                team = normalise(e.get('team', {}).get('name', ''))
                if team == home:
                    h_xg += xg
                else:
                    a_xg += xg
        rows.append({'year': year, 'home_team': home, 'away_team': away, 'home_xg': round(h_xg, 3), 'away_xg': round(a_xg, 3)})
        time.sleep(0.05)
    return pd.DataFrame(rows)

print('Extracting xG 2018...')
sb_2018_df = extract_xg(sb_2018, 2018)
print('Extracting xG 2022...')
sb_2022_df = extract_xg(sb_2022, 2022)
sb_xg = pd.concat([sb_2018_df, sb_2022_df], ignore_index=True)
sb_xg.to_csv(DATA_DIR / 'xg_data.csv', index=False)
print(f'Saved xg_data.csv: {len(sb_xg)} matches')

## 3. Team Features

In [ ]:
def compute_team_features(matches, goals, bookings, sb_xg, teams):
    features = []
    for team in teams:
        home = matches[matches['home_team_name'] == team]
        away = matches[matches['away_team_name'] == team]
        all_m = pd.concat([home, away])
        n = len(all_m)
        if n == 0:
            continue
        wins = (home['home_team_score'] > home['away_team_score']).sum() + (away['away_team_score'] > away['home_team_score']).sum()
        draws = (home['home_team_score'] == home['away_team_score']).sum() + (away['away_team_score'] == away['home_team_score']).sum()
        gf = home['home_team_score'].sum() + away['away_team_score'].sum()
        ga = home['away_team_score'].sum() + away['home_team_score'].sum()
        tournament_wins = ((all_m['stage_name'] == 'Final') & ((all_m['home_team_name'] == team) & (all_m['home_team_score'] > all_m['away_team_score']) | (all_m['away_team_name'] == team) & (all_m['away_team_score'] > all_m['home_team_score']))).sum()
        team_bookings = bookings[bookings['player_team'] == team]
        team_goals = goals[goals['team_name'] == team]
        penalty_goals = (team_goals['penalty'] == True).sum()
        team_sb = sb_xg[(sb_xg['home_team'] == team) | (sb_xg['away_team'] == team)]
        if len(team_sb) > 0:
            h_xg = team_sb[team_sb['home_team'] == team]['home_xg'].sum()
            a_xg = team_sb[team_sb['away_team'] == team]['away_xg'].sum()
            xgf = (h_xg + a_xg) / len(team_sb)
            h_xga = team_sb[team_sb['home_team'] == team]['away_xg'].sum()
            a_xga = team_sb[team_sb['away_team'] == team]['home_xg'].sum()
            xga = (h_xga + a_xga) / len(team_sb)
        else:
            xgf = xga = np.nan
        features.append({
            'team': team,
            'matches': n,
            'win_rate': wins / n,
            'draw_rate': draws / n,
            'avg_goals_for': gf / n,
            'avg_goals_against': ga / n,
            'avg_goal_diff': (gf - ga) / n,
            'avg_xg_scored': round(xgf, 3),
            'avg_xg_conceded': round(xga, 3),
            'tournament_wins': tournament_wins,
            'avg_cards': round(len(team_bookings) / n, 3),
            'penalty_rate': round(penalty_goals / n, 3),
        })
    return pd.DataFrame(features)

team_features = compute_team_features(matches, goals, bookings, sb_xg, ALL_TEAMS_2026)
team_features.to_csv(DATA_DIR / 'team_features.csv', index=False)
print('Saved team_features.csv')
print(team_features[['team', 'matches', 'win_rate', 'avg_goals_for', 'avg_goals_against']].head(10))

## 4. Player Features

In [ ]:
player_goals = goals[goals['goal_type'] != 'Own Goal'].copy()
player_summary = player_goals.groupby(['player_name', 'team_name']).size().reset_index(name='total_goals')
player_summary = player_summary.sort_values('total_goals', ascending=False)
player_summary.to_csv(DATA_DIR / 'player_summary.csv', index=False)
print('Saved player_summary.csv')
print(player_summary.head(15))

## 5. Dark Horse Scores

In [ ]:
FIFA_RANKINGS_2026 = {
    'Mexico': 13, 'South Africa': 57, 'South Korea': 18, 'Czech Republic': 41,
    'Canada': 48, 'Bosnia and Herzegovina': 61, 'Qatar': 50, 'Switzerland': 20,
    'Brazil': 4, 'Morocco': 11, 'Haiti': 90, 'Scotland': 38,
    'United States': 16, 'Paraguay': 60, 'Australia': 22, 'Turkey': 37,
    'Germany': 10, 'Cote dIvoire': 47, 'Thailand': 95, 'Chile': 29,
    'Spain': 8, 'Egypt': 33, 'Tunisia': 62, 'Costa Rica': 49,
    'England': 5, 'Croatia': 7, 'Ghana': 61, 'Panama': 44,
    'France': 2, 'Senegal': 18, 'Iraq': 156, 'Norway': 45,
    'Argentina': 1, 'Algeria': 52, 'Austria': 26, 'Jordan': 96,
    'Portugal': 9, 'DR Congo': 77, 'Uzbekistan': 42, 'Colombia': 12,
    'Netherlands': 6, 'Iran': 20, 'New Zealand': 92, 'Japan': 24,
    'Belgium': 3, 'Uruguay': 25, 'Sweden': 28, 'Saudi Arabia': 55,
}

dh = team_features.copy()
dh['fifa_rank'] = dh['team'].map(FIFA_RANKINGS_2026)
dh = dh[dh['fifa_rank'] > 25].copy()
dh['attack_score'] = (dh['avg_goals_for'] - dh['avg_goals_for'].min()) / (dh['avg_goals_for'].max() - dh['avg_goals_for'].min())
dh['defence_score'] = (dh['avg_goals_against'].max() - dh['avg_goals_against']) / (dh['avg_goals_against'].max() - dh['avg_goals_against'].min())
dh['upset_potential'] = ((dh['fifa_rank'] - 25) / (dh['fifa_rank'].max() - 25)).clip(0, 1)
dh['dark_horse_score'] = 0.35 * dh['attack_score'] + 0.35 * dh['defence_score'] + 0.30 * dh['upset_potential']
dh_save = dh[['team', 'fifa_rank', 'dark_horse_score', 'attack_score', 'defence_score', 'upset_potential']].sort_values('dark_horse_score', ascending=False)
dh_save.to_csv(DATA_DIR / 'dark_horse_scores.csv', index=False)
print('Saved dark_horse_scores.csv')
print('\nTop Dark Horses (rank > 25):')
print(dh_save.head(10).to_string(index=False))

## 6. Complete

In [ ]:
print('\n✅ Data collection complete.')
print('Output files: team_features.csv, player_summary.csv, dark_horse_scores.csv, xg_data.csv')